In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string

In [27]:
# === Download necessary NLTK data ===
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

FINE_TUNED_DIR = '../../../pretained_or_finetune-models'
REVIEWS_DATASET_DIR = '../../dataset'
UTILS_DIR = '../../utils'
NLTK_DATA_PATH = f"{FINE_TUNED_DIR}/nltk_data"

nltk.data.path.append(NLTK_DATA_PATH)

# === Constants ===
STOPWORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()
CONST_VECTOR_SIZE = 50
CONST_DIM_WINDOWS = 5

[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1000)>
[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1000)>
[nltk_data] Error loading wordnet: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1000)>


In [28]:
def advanced_clean_text(text):
    # Define domain-specific stopwords
    custom_stopwords = {'point', 'points', 'interest', 'landmark', 'landmarks', 'site', 'sites'}  # Remove generic tourism terms
    
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = re.sub(r'[^a-z\s]', '', text)  # Remove special characters
    text = re.sub(r'\b(u|ur|b4)\b', 'you', text)  # Replace common abbreviations
    
    words = word_tokenize(text)
    words = [LEMMATIZER.lemmatize(word) for word in words if word not in STOPWORDS]
    words = [word for word in words if word not in custom_stopwords and word not in string.punctuation]  # Remove tourism-related stopwords

    return ' '.join(words)

In [29]:
combined_details_df = pd.read_csv("../data/combined_details.csv")
combined_details_df['tags'] = combined_details_df['tags'].fillna('')
combined_details_df['tags'].replace("", "other", inplace=True)

/var/folders/fk/449tdf_12hzd2r7ps4659q0m0000gn/T/ipykernel_89425/2128574053.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  combined_details_df['tags'].replace("", "other", inplace=True)


In [30]:
combined_details_df['cleaned_tags'] = combined_details_df['tags'].apply(advanced_clean_text)

In [31]:
sentences = combined_details_df['cleaned_tags'].str.split()  # Tokenized tags
word2vec_model = Word2Vec(sentences, vector_size=CONST_VECTOR_SIZE, window=CONST_DIM_WINDOWS, min_count=1, workers=4)

# Compute TF-IDF
tag_texts = [' '.join(tag) for tag in sentences]
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(tag_texts)
tfidf_vocab = tfidf_vectorizer.vocabulary_

In [32]:
def get_tag_vector(tag):
    vectors = []
    for word in tag:
        if word in word2vec_model.wv and word in tfidf_vocab:
            tfidf_weight = tfidf_matrix[0, tfidf_vocab[word]]  # Get TF-IDF weight for the word
            word_vector = word2vec_model.wv[word] * tfidf_weight  # Multiply Word2Vec by TF-IDF weight
            vectors.append(word_vector)
    if vectors:
        return np.mean(vectors, axis=0)  # Average the weighted Word2Vec embeddings
    else:
        return np.zeros(word2vec_model.vector_size)  # Return a zero vector if no valid words

# Generate tag embeddings
combined_details_df['tag_embeddings'] = [get_tag_vector(tag) for tag in sentences]

In [22]:
combined_details_df['tag_embeddings'].iloc[111]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [33]:
combined_details_df = combined_details_df.drop(columns=["Unnamed: 0", "Unnamed: 0.1"])
combined_details_df

,location_id,name,description,web_url,latitude,longitude,website,write_review,rating,rating_image_url,...,rating_4_review_count,rating_5_review_count,tags,trip_types_solo,trip_types_couples,trip_types_business,trip_types_family,trip_types_friends,cleaned_tags,tag_embeddings
0,456633.0,House of Opium,Located in the heart of the infamous Golden Tr...,https://www.tripadvisor.com/Attraction_Review-...,20.351446,100.081660,https://www.tourismthailand.org/Attraction/Hou...,https://www.tripadvisor.com/UserReview-g317131...,4.0,https://www.tripadvisor.com/img/cdsi/img2/rati...,...,363.0,413.0,"History Museums, Specialty Museums",105.0,440.0,9.0,148.0,184.0,history museum specialty museum,"[0.0025900002, -0.0006501009, -0.0038178414, -..."
1,23933132.0,Opium Bar,NaN,https://www.tripadvisor.com/Attraction_Review-...,13.739190,100.508500,http://www.opiumbarbangkok.com/,https://www.tripadvisor.com/UserReview-g293916...,4.0,https://www.tripadvisor.com/img/cdsi/img2/rati...,...,0.0,4.0,Bars & Clubs,0.0,3.0,0.0,0.0,2.0,bar club,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,7099963.0,Wat Phra That Doi Saket,NaN,https://www.tripadvisor.com/Attraction_Review-...,18.872380,99.139930,NaN,https://www.tripadvisor.com/UserReview-g196165...,4.5,https://www.tripadvisor.com/img/cdsi/img2/rati...,...,31.0,44.0,Sacred & Religious Sites,9.0,26.0,0.0,14.0,21.0,sacred religious,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,1390125.0,Mae Ping River Cruise,Leave from the jetty at Wat Chai Mongkol and t...,https://www.tripadvisor.com/Attraction_Review-...,18.780819,99.005210,http://www.maepingrivercruise.com/,https://www.tripadvisor.com/UserReview-g293917...,3.5,https://www.tripadvisor.com/img/cdsi/img2/rati...,...,127.0,102.0,Boat Tours,31.0,186.0,2.0,43.0,57.0,boat tour,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,547675.0,Pai River,NaN,https://www.tripadvisor.com/Attraction_Review-...,19.359318,98.445305,NaN,https://www.tripadvisor.com/UserReview-g303916...,4.0,https://www.tripadvisor.com/img/cdsi/img2/rati...,...,58.0,52.0,Bodies of Water,30.0,56.0,1.0,15.0,46.0,body water,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1761,10053134.0,Chao Lao Beach,NaN,https://www.tripadvisor.com/Attraction_Review-...,12.533703,101.940470,NaN,https://www.tripadvisor.com/UserReview-g223726...,4.0,https://www.tripadvisor.com/img/cdsi/img2/rati...,...,25.0,19.0,Beaches,3.0,18.0,1.0,13.0,11.0,beach,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1762,23744675.0,Chao Lao Fish Bridge,NaN,https://www.tripadvisor.com/Attraction_Review-...,12.571852,101.923390,NaN,https://www.tripadvisor.com/UserReview-g323879...,4.0,https://www.tripadvisor.com/img/cdsi/img2/rati...,...,1.0,0.0,Bridges,0.0,0.0,0.0,0.0,0.0,bridge,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1763,10266462.0,Wat Burapa Temple,NaN,https://www.tripadvisor.com/Attraction_Review-...,15.230770,104.865680,NaN,https://www.tripadvisor.com/UserReview-g656472...,3.5,https://www.tripadvisor.com/img/cdsi/img2/rati...,...,1.0,0.0,"Sacred & Religious Sites, Architectural Buildings",1.0,0.0,1.0,0.0,2.0,sacred religious architectural building,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1764,12881708.0,Kung Wiman Beach,NaN,https://www.tripadvisor.com/Attraction_Review-...,12.608021,101.870415,NaN,https://www.tripadvisor.com/UserReview-g223726...,4.0,https://www.tripadvisor.com/img/cdsi/img2/rati...,...,7.0,5.0,Beaches,1.0,4.0,0.0,4.0,3.0,beach,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [34]:
combined_details_df.to_csv("input/tag_embeddings.csv", index=False)

In [10]:
df = pd.DataFrame(tag_embeddings)
df.to_parquet("input/tag_embeddings.parquet", index=False)
df

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,0.002915,-0.001217,-0.003961,-0.001022,0.008909,0.006981,-0.001817,0.007706,-0.007217,0.004565,...,-0.003344,-0.002406,-0.002017,0.005441,0.006196,0.005129,-0.000977,-0.00574,-0.002476,-0.000078
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1761,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
1762,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
1763,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
1764,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000


In [14]:
sentences

0               [history, museum, specialty, museum]
1                                        [bar, club]
2                                [sacred, religious]
3                                       [boat, tour]
4                                      [body, water]
                            ...                     
1761                                         [beach]
1762                                        [bridge]
1763    [sacred, religious, architectural, building]
1764                                         [beach]
1765                                       [lookout]
Name: cleaned_tags, Length: 1766, dtype: object

In [13]:
pd.DataFrame(sentences).to_parquet("input/tag_sentences.parquet", index=False)